In [1]:
# Zelle 1 — Setup, komplett eigenständig

import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Lasso
from sklearn.ensemble import HistGradientBoostingRegressor

training_data = pd.read_csv("train.csv")
testing_data = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

train_indexed = training_data.set_index("Id")
y_full = train_indexed["SalePrice"]
X_full_raw = train_indexed.drop(columns="SalePrice")

def add_features(df):
    df = df.copy()

    df["MSSubClass"] = df["MSSubClass"].astype("string")
    df["Functional"] = df["Functional"].fillna("Typ")
    df["KitchenQual"] = df["KitchenQual"].fillna("TA")
    df["GarageYrBlt"] = df["GarageYrBlt"].fillna(df["YearBuilt"])

    df["HasGarage"] = (
        (df["GarageArea"].fillna(0) > 0)
        | (df["GarageCars"].fillna(0) > 0)
        | df["GarageType"].notna()
    ).astype(int)
    df["HasBasement"]  = (df["TotalBsmtSF"].fillna(0) > 0).astype(int)
    df["HasFireplace"] = (df["Fireplaces"].fillna(0) > 0).astype(int)
    df["HasPool"]      = (df["PoolArea"].fillna(0) > 0).astype(int)
    has_masonry_type = df["MasVnrType"].notna() & df["MasVnrType"].ne("None")
    df["HasMasonry"] = (has_masonry_type | df["MasVnrArea"].fillna(0).gt(0)).astype(int)
    df["MasVnrArea"] = df["MasVnrArea"].fillna(0)
    df["LotFrontageMissing"] = df["LotFrontage"].isna().astype(int)

    df["HouseAge"]          = df["YrSold"] - df["YearBuilt"]
    df["YearsSinceRemodel"] = (df["YrSold"] - df["YearRemodAdd"]).clip(lower=0)
    df["TotalSF"]           = df["TotalBsmtSF"].fillna(0) + df["1stFlrSF"] + df["2ndFlrSF"]
    df["TotalBathrooms"]    = df["FullBath"] + 0.5*df["HalfBath"] + df["BsmtFullBath"].fillna(0) + 0.5*df["BsmtHalfBath"].fillna(0)

    return df

ordinal_categories = {
    "ExterQual":   ["None","Po","Fa","TA","Gd","Ex"],
    "ExterCond":   ["None","Po","Fa","TA","Gd","Ex"],
    "BsmtQual":    ["None","Po","Fa","TA","Gd","Ex"],
    "BsmtCond":    ["None","Po","Fa","TA","Gd","Ex"],
    "HeatingQC":   ["None","Po","Fa","TA","Gd","Ex"],
    "KitchenQual": ["Po","Fa","TA","Gd","Ex"],
    "FireplaceQu": ["None","Po","Fa","TA","Gd","Ex"],
    "GarageQual":  ["None","Po","Fa","TA","Gd","Ex"],
    "GarageCond":  ["None","Po","Fa","TA","Gd","Ex"],
    "PoolQC":      ["None","Po","Fa","TA","Gd","Ex"],
    "BsmtExposure":["None","No","Mn","Av","Gd"],
    "BsmtFinType1":["None","Unf","LwQ","Rec","BLQ","ALQ","GLQ"],
    "BsmtFinType2":["None","Unf","LwQ","Rec","BLQ","ALQ","GLQ"],
    "PavedDrive":  ["N","P","Y"],
    "LandSlope":   ["Sev","Mod","Gtl"],
    "GarageFinish":["None","Unf","RFn","Fin"],
    "Functional":  ["Sal","Sev","Maj2","Maj1","Mod","Min2","Min1","Typ"],
    "Fence":       ["None","MnWw","GdWo","MnPrv","GdPrv"],
}
ordinal_features = list(ordinal_categories.keys())

X_full = add_features(X_full_raw)
X_competition = add_features(testing_data.set_index("Id"))
y_full_log = np.log1p(y_full)

num_features = X_full.select_dtypes(include="number").columns
cat_features = X_full.select_dtypes(exclude="number").columns
nominal_features = [c for c in cat_features if c not in ordinal_features]

linear_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ]), num_features),
    ("ordinal", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
        ("encoder", OrdinalEncoder(categories=list(ordinal_categories.values()), handle_unknown="use_encoded_value", unknown_value=-1)),
        ("scaler", RobustScaler())
    ]), ordinal_features),
    ("nominal", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="N_A")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), nominal_features),
])

tree_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), num_features),
    ("ordinal", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
        ("encoder", OrdinalEncoder(categories=list(ordinal_categories.values()), handle_unknown="use_encoded_value", unknown_value=-1))
    ]), ordinal_features),
    ("nominal", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="N_A")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), nominal_features),
])

lasso_model = Pipeline([("preprocessor", linear_preprocessor), ("regressor", Lasso(alpha=0.001, max_iter=50000))])

histgb_model = Pipeline([("preprocessor", tree_preprocessor), ("regressor", HistGradientBoostingRegressor(
    random_state=123, min_samples_leaf=3, max_leaf_nodes=10, max_iter=200, learning_rate=0.05
))])

In [2]:
# Zelle 2 — Lasso final fitten, Submission 1

lasso_final = lasso_model.fit(X_full, y_full_log)
lasso_predictions = np.expm1(lasso_final.predict(X_competition))

submission_1 = sample_submission[["Id"]].copy()
submission_1["SalePrice"] = submission_1["Id"].map(pd.Series(lasso_predictions, index=X_competition.index))

assert submission_1["SalePrice"].notna().all()
assert (submission_1["SalePrice"] > 0).all()

print(submission_1.loc[submission_1["Id"] == 2550])  # Sanity-Check: Landet Id 2550 nicht bei 600k+?
print(submission_1.head())
print(submission_1["SalePrice"].describe())

submission_1.to_csv("submission_1_lasso.csv", index=False)

        Id      SalePrice
1089  2550  840058.919566
     Id      SalePrice
0  1461  114062.760328
1  1462  145527.033726
2  1463  171186.062745
3  1464  195917.723657
4  1465  196570.195481
count      1459.000000
mean     177608.567923
std       74628.035937
min       44171.894528
25%      126738.157051
50%      157927.744719
75%      209976.893945
max      840058.919566
Name: SalePrice, dtype: float64


In [3]:
# Zelle 3 — HistGB v2 final fitten, Submission 2

histgb_final = histgb_model.fit(X_full, y_full_log)
histgb_predictions = np.expm1(histgb_final.predict(X_competition))

submission_2 = sample_submission[["Id"]].copy()
submission_2["SalePrice"] = submission_2["Id"].map(pd.Series(histgb_predictions, index=X_competition.index))

assert submission_2["SalePrice"].notna().all()
assert (submission_2["SalePrice"] > 0).all()

print(submission_2.loc[submission_2["Id"] == 2550])  # Sanity-Check, erwartet deutlich vorsichtiger als Lasso
print(submission_2.head())
print(submission_2["SalePrice"].describe())

submission_2.to_csv("submission_2_histgb.csv", index=False)

        Id      SalePrice
1089  2550  235266.226205
     Id      SalePrice
0  1461  122407.101484
1  1462  166467.588062
2  1463  180167.248013
3  1464  190494.159158
4  1465  184930.354157
count      1459.000000
mean     177768.765187
std       76950.499481
min       43180.751181
25%      127377.273629
50%      156709.513696
75%      205917.381413
max      572202.362276
Name: SalePrice, dtype: float64


In [4]:
# Zelle 4 — Submission 3, Blend im Log-Raum, mit bereits gefitteten Modellen

lasso_log_pred = lasso_final.predict(X_competition)
histgb_log_pred = histgb_final.predict(X_competition)

blend_log_pred = 0.7 * histgb_log_pred + 0.3 * lasso_log_pred
blend_saleprice = np.maximum(np.expm1(blend_log_pred), 0)

submission_3 = sample_submission[["Id"]].copy()
submission_3["SalePrice"] = submission_3["Id"].map(pd.Series(blend_saleprice, index=X_competition.index))

assert len(submission_3) == 1459
assert submission_3["SalePrice"].notna().all()
assert (submission_3["SalePrice"] >= 0).all()

print(submission_3.loc[submission_3["Id"] == 2550])  # sollte zwischen Lassos 840k und HistGBs 235k liegen
print(submission_3["SalePrice"].describe())

submission_3.to_csv("submission_3_blend_70_histgb_30_lasso.csv", index=False)

        Id      SalePrice
1089  2550  344655.189253
count      1459.000000
mean     177556.086452
std       75352.305487
min       43475.736079
25%      127125.012410
50%      156515.762921
75%      208315.088793
max      552558.010419
Name: SalePrice, dtype: float64


In [5]:
# Zelle 5 — Lasso mit Log-Features, final gefittet, Submission 4

skewed_features = ["GrLivArea", "LotArea", "1stFlrSF", "TotalSF", "TotalBsmtSF"]

def add_log_features(df):
    df = df.copy()
    for col in skewed_features:
        df[col] = np.log1p(df[col])
    return df

X_full_logfeat = add_log_features(X_full)
X_competition_logfeat = add_log_features(X_competition)

lasso_logfeat_final = lasso_model.fit(X_full_logfeat, y_full_log)
lasso_logfeat_predictions = np.expm1(lasso_logfeat_final.predict(X_competition_logfeat))

submission_4 = sample_submission[["Id"]].copy()
submission_4["SalePrice"] = submission_4["Id"].map(pd.Series(lasso_logfeat_predictions, index=X_competition_logfeat.index))

assert submission_4["SalePrice"].notna().all()
assert (submission_4["SalePrice"] > 0).all()

print(submission_4.loc[submission_4["Id"] == 2550])  # der eigentliche Test unserer Hypothese von eben
print(submission_4["SalePrice"].describe())

submission_4.to_csv("submission_4_lasso_logfeatures.csv", index=False)

        Id      SalePrice
1089  2550  562775.442966
count      1459.000000
mean     177550.875817
std       72662.086284
min       43197.994284
25%      126864.818942
50%      158542.751311
75%      212069.705709
max      562775.442966
Name: SalePrice, dtype: float64


In [6]:
# Zelle 6 — HistGB + Log-Feature-Lasso, 50/50, Submission 5

blend_log_pred_v2 = 0.5 * histgb_log_pred + 0.5 * lasso_logfeat_final.predict(X_competition_logfeat)
blend_saleprice_v2 = np.maximum(np.expm1(blend_log_pred_v2), 0)

submission_5 = sample_submission[["Id"]].copy()
submission_5["SalePrice"] = submission_5["Id"].map(pd.Series(blend_saleprice_v2, index=X_competition_logfeat.index))

assert submission_5["SalePrice"].notna().all()
assert (submission_5["SalePrice"] > 0).all()

print(submission_5.loc[submission_5["Id"] == 2550])
print(submission_5["SalePrice"].describe())

submission_5.to_csv("submission_5_blend_50_50_logfeat.csv", index=False)

        Id     SalePrice
1089  2550  363871.02793
count      1459.000000
mean     177518.345012
std       74162.473677
min       43189.371872
25%      126662.781321
50%      157224.640491
75%      209231.578192
max      524050.634049
Name: SalePrice, dtype: float64


In [7]:
# Zelle 7 — Submission 6: 0.5 × XGBoost + 0.5 × Lasso (Log-Features), gemischt im Log-Raum
from sklearn.base import clone
from xgboost import XGBRegressor

X_full_logfeat = add_log_features(X_full)
X_competition_logfeat = add_log_features(X_competition)

lasso_log = clone(lasso_model).fit(X_full_logfeat, y_full_log)

xgb_model = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("regressor", XGBRegressor(
        n_estimators=600, learning_rate=0.03, max_depth=3, subsample=0.8,
        colsample_bytree=0.8, random_state=123, n_jobs=-1)),
])
xgb_final = xgb_model.fit(X_full, y_full_log)

blend6_log_pred = 0.5 * xgb_final.predict(X_competition) + 0.5 * lasso_log.predict(X_competition_logfeat)
blend6_saleprice = np.maximum(np.expm1(blend6_log_pred), 0)

submission_6 = sample_submission[["Id"]].copy()
submission_6["SalePrice"] = submission_6["Id"].map(pd.Series(blend6_saleprice, index=X_competition.index))

assert len(submission_6) == 1459
assert submission_6["SalePrice"].notna().all()
assert (submission_6["SalePrice"] > 0).all()

print(submission_6.loc[submission_6["Id"] == 2550])
print(submission_6["SalePrice"].describe())

submission_6.to_csv("submission_6_blend_50_50_xgboost_lasso_logfeat.csv", index=False)

        Id      SalePrice
1089  2550  361935.780896
count      1459.000000
mean     177396.440229
std       73761.677308
min       42130.810753
25%      127230.560918
50%      156886.895165
75%      209354.848332
max      496643.499766
Name: SalePrice, dtype: float64
